In [2]:
print("hello world")

hello world


In [3]:
%pwd

'x:\\LLMRAG\\llm'

In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import pipeline

x:\LLMRAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
os.chdir("../")
%pwd

'x:\\LLMRAG'

In [6]:
# 1. Load PDFs
def load_pdf_files(path):
    loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

In [7]:
# 2. Split documents
def split_docs(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    return splitter.split_documents(documents)

In [8]:
# 3. Embeddings
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

In [9]:
# 4. Vector store
def create_vectorstore(docs, embeddings):
    return FAISS.from_documents(docs, embeddings)

In [10]:
# 5. LLM
def get_llm():
    pipe = pipeline(
        "text2text-generation",
        model="google/flan-t5-base",
        max_length=512
    )
    return HuggingFacePipeline(pipeline=pipe)

In [11]:
# 6.MANUAL RAG FUNCTION (KEY PART)
def ask_question(query, vectorstore, llm):
    # Step 1: Retrieve relevant docs
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(query)

    # Step 2: Combine context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Step 3: Create prompt
    template = f"""
    You are a professional medical assistant. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer based on the context, just say that you don't know, don't try to make up an answer.
    Keep the answer concise and professional.

    Context: {context}

    Question: {query}

    Answer:
    """
    prompt = template.strip()

    # Step 4: Get LLM response
    response = llm.invoke(prompt)

    return response


# =========================
# RUN PIPELINE
# =========================

docs = load_pdf_files("data")
chunks = split_docs(docs)

embeddings = get_embeddings()
vectorstore = create_vectorstore(chunks, embeddings)

llm = get_llm()

# Ask question
response = ask_question("i have a fever and clod what i can do?", vectorstore, llm)
print(response)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1908.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
# Ask question
response = ask_question("i have a fever and clod what i can do?", vectorstore, llm)
print(response)